In [2]:
import os
import random
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from faker import Faker

# Set random seed for reproducibility
fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

print("Starting synthetic data generation...")

# Define output directory
os.makedirs("../data", exist_ok=True)

# ---------------------------------------------------------
# 1. Dim_Companies
# ---------------------------------------------------------
num_companies = 1000
industries = ["Fintech", "Edtech", "E-commerce", "Healthcare", "Logistics", "Cybersecurity"]
company_sizes = ["SMB", "Mid-Market", "Enterprise"]
tiers = ["Starter", "Pro", "Enterprise"]

start_window = datetime(2024, 1, 1)
end_window = datetime(2026, 9, 30)

companies = []
for cid in range(101, 101 + num_companies):
    signup = fake.date_between(start_date=start_window, end_date=datetime(2025, 12, 31))
    companies.append({
        "company_id": cid,
        "company_name": fake.company(),
        "industry": random.choice(industries),
        "company_size": random.choices(company_sizes, weights=[0.55, 0.35, 0.10])[0],
        "signup_date": signup
    })

df_companies = pd.DataFrame(companies)
df_companies.to_csv("../data/dim_companies.csv", index=False)
print("Saved: dim_companies.csv")

# ---------------------------------------------------------
# 2. Dim_Subscriptions
# ---------------------------------------------------------
subscriptions = []
churn_reasons = ["Price Hike", "Competitor Switch", "Low Feature Adoption", "Budget Cut", "Poor Support Response"]

for _, row in df_companies.iterrows():
    cid = row["company_id"]
    plan = random.choices(tiers, weights=[0.45, 0.40, 0.15])[0]
    
    if plan == "Starter":
        mrr = random.randint(150, 400)
    elif plan == "Pro":
        mrr = random.randint(500, 1500)
    else:
        mrr = random.randint(2000, 6000)
        
    start_dt = row["signup_date"]
    
    is_churned = random.random() < 0.25
    if is_churned:
        days_active = random.randint(60, 450)
        churn_dt = start_dt + timedelta(days=days_active)
        if churn_dt > end_window.date():
            is_churned = False
            churn_dt = None
            reason = None
        else:
            reason = random.choice(churn_reasons)
    else:
        churn_dt = None
        reason = None
        
    subscriptions.append({
        "subscription_id": 5000 + cid,
        "company_id": cid,
        "plan_name": plan,
        "mrr": mrr,
        "arr": mrr * 12,
        "start_date": start_dt,
        "churn_date": churn_dt,
        "churn_status": "Churned" if is_churned else "Active",
        "churn_reason": reason
    })

df_subscriptions = pd.DataFrame(subscriptions)
df_subscriptions.to_csv("../data/dim_subscriptions.csv", index=False)
print("Saved: dim_subscriptions.csv")

# ---------------------------------------------------------
# 3. Fact_SupportTickets
# ---------------------------------------------------------
tickets = []
ticket_id = 80001
priorities = ["Low", "Medium", "High", "Critical"]

for _, row in df_subscriptions.iterrows():
    cid = row["company_id"]
    is_churned = row["churn_status"] == "Churned"
    
    num_tix = random.randint(2, 8) if is_churned else random.randint(0, 4)
    
    for _ in range(num_tix):
        t_date = fake.date_between(start_date=row["start_date"], end_date=row["churn_date"] if is_churned else end_window.date())
        priority = random.choices(priorities, weights=[0.3, 0.4, 0.2, 0.1])[0]
        
        csat = random.choices([1, 2, 3, 4, 5], weights=[0.4, 0.3, 0.15, 0.1, 0.05])[0] if is_churned else random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.1, 0.2, 0.35, 0.3])[0]
        
        tickets.append({
            "ticket_id": ticket_id,
            "company_id": cid,
            "ticket_date": t_date,
            "priority": priority,
            "resolution_time_hrs": round(random.uniform(4.0, 72.0), 1),
            "csat_score": csat
        })
        ticket_id += 1

df_tickets = pd.DataFrame(tickets)
df_tickets.to_csv("../data/fact_support_tickets.csv", index=False)
print("Saved: fact_support_tickets.csv")

# ---------------------------------------------------------
# 4. Fact_DailyUsage
# ---------------------------------------------------------
usage_records = []
usage_id = 1

tracking_dates = pd.date_range(start="2026-01-01", end="2026-09-30", freq="W-MON")

for _, sub in df_subscriptions.iterrows():
    cid = sub["company_id"]
    is_churned = sub["churn_status"] == "Churned"
    churn_dt = pd.to_datetime(sub["churn_date"]) if is_churned else None
    
    base_activity = 100 if sub["plan_name"] == "Enterprise" else (50 if sub["plan_name"] == "Pro" else 15)
    
    for dt in tracking_dates:
        if dt.date() < sub["start_date"]:
            continue
        if is_churned and dt > churn_dt:
            continue
            
        decay = 1.0
        if is_churned and (churn_dt - dt).days < 60:
            decay = 0.25
            
        active_users = max(1, int(np.random.normal(base_activity * decay, base_activity * 0.15)))
        logins = active_users * random.randint(3, 7)
        feature_clicks = logins * random.randint(10, 25)
        
        usage_records.append({
            "usage_id": usage_id,
            "company_id": cid,
            "usage_date": dt.date(),
            "active_users": active_users,
            "total_logins": logins,
            "feature_clicks": feature_clicks
        })
        usage_id += 1

df_usage = pd.DataFrame(usage_records)
df_usage.to_csv("../data/fact_daily_usage.csv", index=False)
print("Saved: fact_daily_usage.csv")

print("Pipeline execution complete: All datasets generated successfully.")

Starting synthetic data generation...
Saved: dim_companies.csv
Saved: dim_subscriptions.csv
Saved: fact_support_tickets.csv
Saved: fact_daily_usage.csv
Pipeline execution complete: All datasets generated successfully.
